In [1]:
using LinearAlgebra, MPSKit, TensorKit, MPSKitExperimental
LinearAlgebra.BLAS.set_num_threads(1);

file = "N2.CCPVDZ.FCIDUMP"
D = 50
sweeps = 10

# ERI is the Electronic Repulsion Integral, K is the two-body term, E0 is an energy shift,
# NORB is the number of orbitals (mps length), NELEC is the number of electrons and MS2 is the total spin.
# fcidump files can contain orbital symmetries, but these are ignored.
(ERI, K, E0, NORB, NELEC, MS2) = parse_fcidump(file)

# the second return value contains some metadata which can be ignored
(ham, _) = fused_quantum_chemistry_hamiltonian(E0, K, ERI, Float64);

psp = Vect[(Irrep[U₁] ⊠ Irrep[SU₂] ⊠ FermionParity)]((0, 0, 0) => 1, (1, 1 // 2, 1) => 1, (2, 0, 0) => 1);
left = Vect[(Irrep[U₁] ⊠ Irrep[SU₂] ⊠ FermionParity)]((-NELEC, MS2 // 2, mod(-NELEC, 2)) => 1);
right = oneunit(left);

virtual = Vect[(Irrep[U₁] ⊠ Irrep[SU₂] ⊠ FermionParity)]((i, s, b) => 1 for i in -NELEC:0, s in 0:(1 // 2):(MS2 // 2 + 1), b in (0, 1));

st = FiniteMPS(rand, Float64, NORB, psp, virtual; left, right);

# environments that live on disk, instead of in ram
envs = disk_environments(st, ham);

(st, envs) = find_groundstate(st, ham, DMRG2(; trscheme = truncrank(D), verbosity = 1000, maxiter = sweeps, miniter = sweeps), envs);

println("E = ", real(expectation_value(st, ham, envs)))


┌ Info: DMRG2   1:	obj = -1.089438760100e+02	err = 9.9999998619e-01	time = 1.85 min
└ @ MPSKit /home/maarten/projects/MPSKit.jl/src/algorithms/groundstate/dmrg.jl:197
┌ Info: DMRG2   2:	obj = -1.090744729458e+02	err = 1.0112935717e-02	time = 37.27 sec
└ @ MPSKit /home/maarten/projects/MPSKit.jl/src/algorithms/groundstate/dmrg.jl:197
┌ Info: DMRG2   3:	obj = -1.091364249226e+02	err = 3.5809790841e-03	time = 42.37 sec
└ @ MPSKit /home/maarten/projects/MPSKit.jl/src/algorithms/groundstate/dmrg.jl:197
┌ Info: DMRG2   4:	obj = -1.091417551982e+02	err = 1.8418968572e-04	time = 38.84 sec
└ @ MPSKit /home/maarten/projects/MPSKit.jl/src/algorithms/groundstate/dmrg.jl:197
┌ Info: DMRG2   5:	obj = -1.091435165357e+02	err = 7.9197051365e-05	time = 45.43 sec
└ @ MPSKit /home/maarten/projects/MPSKit.jl/src/algorithms/groundstate/dmrg.jl:197
┌ Info: DMRG2   6:	obj = -1.091437880567e+02	err = 8.1104933323e-06	time = 43.75 sec
└ @ MPSKit /home/maarten/projects/MPSKit.jl/src/algorithms/groundstate/dmrg.

E = -109.14436579375061


┌ Warning: DMRG2 cancel 10:	obj = -1.091443657938e+02	err = 5.6012529069e-06	time = 8.72 min
└ @ MPSKit /home/maarten/projects/MPSKit.jl/src/algorithms/groundstate/dmrg.jl:195
